In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

In [ ]:
df = pd.read_csv('cafe_reviews_filtered_final.csv')


FileNotFoundError: [Errno 2] No such file or directory: 'cafe_reviews_filtered_final.csv'

In [ ]:
print("컬럼 목록:")
print(df.columns.tolist())

print("\n결측치:")
print(df.isnull().sum())

print("\n데이터 크기:")
print(f"행: {df.shape[0]}")
print(f"열: {df.shape[1]}")

In [ ]:
#[중복리뷰확인]1.완전 동일한 행
full_duplicates = df.duplicated().sum()

print(f"완전히 동일한 행 수: {full_duplicates}")

In [ ]:
#[중복리뷰확인]2.같은 카페의 같은 리뷰
content_duplicates = df.duplicated(
    subset=['name', 'full_text']
).sum()

print(f"같은 카페 + 동일한 리뷰 수: {content_duplicates}")

In [ ]:
#[중복리뷰확인]
duplicate_rows = df[
    df.duplicated(
        subset=['name', 'full_text'],
        keep=False
    )
].sort_values(['name', 'full_text'])

duplicate_rows[['name', 'full_text']]

In [ ]:
#[중복리뷰확인]****제거****
df_no_duplicate = df.drop_duplicates(
    subset=['name', 'full_text']
).copy()

print("중복 제거 전:", df.shape)
print("중복 제거 후:", df_no_duplicate.shape)

In [ ]:
#글자수너무짧은리뷰확인
df['text_length'] = df['full_text'].astype(str).str.len()

df['text_length'].describe()

In [ ]:
#실제분석에 사용할 단어수
df['token_count'] = (
    df['cleaned_tokens']
    .astype(str)
    .str.split()
    .str.len()
)

df['token_count'].describe()

In [ ]:
#리뷰길이 분포 시각화
plt.figure(figsize=(10, 5))

plt.hist(df['token_count'], bins=50)

plt.title('Distribution of Review Token Count')
plt.xlabel('Number of Tokens')
plt.ylabel('Number of Reviews')

plt.show()

In [ ]:
#짧은 리뷰 기준 확인
thresholds = [1, 5, 10, 20, 25]

for threshold in thresholds:

    count = (df['token_count'] <= threshold).sum()
    ratio = count / len(df) * 100

    print(
        f"{threshold}개 이하 토큰 리뷰: "
        f"{count}개 ({ratio:.2f}%)"
    )

In [ ]:
#10개 이하 확인
short_reviews = df[df['token_count'] <= 10]

print(f"짧은 리뷰 수: {len(short_reviews)}")

short_reviews[
    ['name', 'full_text', 'cleaned_tokens', 'token_count']
].head(20)

In [ ]:
#****10개**** 이하 제거
df_clean = df[df['token_count'] > 10].copy()

print("짧은 리뷰 제거 전:", df.shape)
print("짧은 리뷰 제거 후:", df_clean.shape)

In [ ]:
#카페별 리뷰수
cafe_review_count = (
    df.groupby('name')
    .size()
    .sort_values(ascending=False)
)

cafe_review_count

In [ ]:
print(cafe_review_count.describe())

In [ ]:
# 나눔 고딕 폰트 설치
%cd /content
!wget https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = 'NanumGothic-Regular.ttf'       # 설치한 폰트 경로
fm.fontManager.addfont(font_path)   # 폰트 경로 추가

plt.rcParams['font.family'] = 'NanumGothic' # 사용 폰트 입력
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호 사용

In [ ]:
#카페별 리뷰수 시각화
plt.figure(figsize=(10, 12))

cafe_review_count.sort_values().plot(kind='barh')

plt.title('Number of Reviews by Cafe')
plt.xlabel('Number of Reviews')
plt.ylabel('Cafe')

plt.show()

In [ ]:
#리뷰수(데이터양)가 적은 카페 분류
thresholds = [10, 20, 30, 50]

for threshold in thresholds:

    count = (cafe_review_count < threshold).sum()

    print(
        f"리뷰 {threshold}개 미만 카페: "
        f"{count}개"
    )

In [ ]:
#리뷰수(데이터양)가 적은 카페 목록(****30개이하****)
min_review_threshold = 30

small_cafes = cafe_review_count[
    cafe_review_count < min_review_threshold
]

print(f"리뷰 {min_review_threshold}개 미만 카페 수: {len(small_cafes)}")

small_cafes

In [ ]:
# 리뷰수(데이터양)가 적은 카페 제거(****30개미만제거****)
cafes_to_remove = small_cafes.index.tolist()
df_final = df_clean[~df_clean['name'].isin(cafes_to_remove)].copy()

print("리뷰 30개 미만 카페 제거 전:", df_clean.shape)
print("리뷰 30개 미만 카페 제거 후:", df_final.shape)

In [ ]:
# 리뷰 30개 미만 카페 제거 후 리뷰수 재계산
cafe_review_count_final = (
    df_final.groupby('name')
    .size()
    .sort_values(ascending=False)
)

# 리뷰수 불균형 편차 확인(**CV:0.68로 어느정도편차 존재)
mean_reviews = cafe_review_count_final.mean()
std_reviews = cafe_review_count_final.std()
cv = std_reviews_final / mean_reviews_final

print(f"제거 후 평균 리뷰 수: {mean_reviews:.2f}")
print(f"제거 후 표준편차: {std_reviews:.2f}")
print(f"제거 후 변동계수(CV): {cv:.2f}")

In [ ]:
#불용어 후보 추출
all_tokens = []

for text in df['cleaned_tokens'].dropna():

    tokens = str(text).split()
    all_tokens.extend(tokens)

In [ ]:
#단어 빈도 계산
word_counts = Counter(all_tokens)

word_freq = pd.DataFrame(
    word_counts.most_common(),
    columns=['word', 'frequency']
)

word_freq.head(30)

In [ ]:
#상위단어 50개 확인
word_freq.head(50)

In [ ]:
#카페별 특정토큰 등장 여부 알아보기
cafe_count = df['name'].nunique()

word_cafe_count = {}

for cafe, group in df.groupby('name'):

    cafe_tokens = set(
        ' '.join(
            group['cleaned_tokens']
            .dropna()
            .astype(str)
        ).split()
    )

    for word in cafe_tokens:
        word_cafe_count[word] = word_cafe_count.get(word, 0) + 1

In [ ]:
word_cafe_df = pd.DataFrame(
    list(word_cafe_count.items()),
    columns=['word', 'cafe_count']
)

In [ ]:
word_analysis = word_freq.merge(
    word_cafe_df,
    on='word',
    how='left'
)

word_analysis['cafe_ratio'] = (
    word_analysis['cafe_count']
    / cafe_count
)

word_analysis.head(30)

In [ ]:
#불용어 후보 추출(**기준:전체 빈도가 높고 & 70% 이상 카페에서 등장)
stopword_candidates = word_analysis[
    (word_analysis['frequency'] >= 50) &
    (word_analysis['cafe_ratio'] >= 0.7)
].sort_values(
    'frequency',
    ascending=False
)

stopword_candidates.head(50)

In [ ]:
#카페이름 포함 확인
cafe_names = df['name'].unique()

print(cafe_names)

In [ ]:
# 카페 이름 토큰 제거 함수 (개선된 로직)
def remove_cafe_name_parts_from_tokens(row):
    cafe_name = str(row['name'])
    tokens = str(row['cleaned_tokens']).split()

    filtered_tokens = []
    for token in tokens:
        # 토큰이 카페 이름 문자열의 일부이거나, 카페 이름 문자열이 토큰의 일부인 경우를 제거
        # 예: cafe_name='메모러블모먼트', token='메모' -> '메모' in '메모러블모먼트' = True, 제거
        # 예: cafe_name='벌스커피', token='벌스커피' -> '벌스커피' in '벌스커피' = True, 제거
        # 예: cafe_name='카페 오텀', token='카페' -> '카페' in '카페 오텀' = True, 제거 (의도된 동작)
        if not (token in cafe_name or cafe_name in token):
            filtered_tokens.append(token)
    return ' '.join(filtered_tokens)

# df_final 데이터프레임에 적용
df_final['cleaned_tokens_no_name'] = df_final.apply(remove_cafe_name_parts_from_tokens, axis=1)

# 새로운 token_count 컬럼 생성
df_final['token_count_no_name'] = df_final['cleaned_tokens_no_name'].str.split().str.len()

print("카페 이름 토큰 제거 전 (df_final의 첫 5개 행):")
display(df_final[['name', 'cleaned_tokens', 'token_count']].head())

print("\n카페 이름 토큰 제거 후 (df_final의 첫 5개 행):")
display(df_final[['name', 'cleaned_tokens_no_name', 'token_count_no_name']].head())

In [ ]:
stopword_candidates.to_csv(
    "stopword_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("불용어 후보 파일 저장 완료!")

In [ ]:
display(df_final.head(30))

In [ ]:
stopword_df = pd.read_csv('stopword_candidates_final.csv', encoding='cp949')
stopwords = stopword_df[stopword_df['stopword'] == 1]['word'].tolist()

In [ ]:
stopword_df = pd.read_csv('stopword_candidates_final.csv', encoding='cp949')
stopwords = stopword_df[stopword_df['stopword'] == 1]['word'].tolist()

In [ ]:
def remove_stopwords(tokens_str, stopwords_list):
    tokens = str(tokens_str).split()
    filtered_tokens = [token for token in tokens if token not in stopwords_list]
    return ' '.join(filtered_tokens)

df_final['cleaned_tokens_final'] = df_final['cleaned_tokens_no_name'].apply(lambda x: remove_stopwords(x, stopwords))
df_final['token_count_final'] = df_final['cleaned_tokens_final'].str.split().str.len()

print("불용어 제거 후 (df_final의 첫 5개 행):")
display(df_final[['name', 'cleaned_tokens_no_name', 'token_count_no_name', 'cleaned_tokens_final', 'token_count_final']].head())

In [ ]:
import pandas as pd

# Load df_final from CSV to ensure it's defined
# Assuming df_final.csv contains all necessary columns up to 'cleaned_tokens_final'
df_final = pd.read_csv('df_final.csv')

location_stopwords = ['맛집', '올림픽', '공원', '강동구청', '강동구청역',
                      '천호역', '둔촌동역', '성내로','주차','주차장','건물',
                      '양재대로','강동역','동네','출구','골목','인근','둔촌',
                      '막상','무조건','올림픽공원역','한국','체대','둔촌역','한체대',
                      '교회','오륜','체육','대학교','아파트','스포츠','거리','성내']

def remove_location_stopwords(tokens_str, stopwords_list):
    # Ensure tokens_str is treated as a string, then split
    tokens = str(tokens_str).split()
    filtered_tokens = [token for token in tokens if token not in stopwords_list]
    return ' '.join(filtered_tokens)

df_final['cleaned_tokens_analysis'] = df_final['cleaned_tokens_final'].apply(lambda x: remove_location_stopwords(x, location_stopwords))
df_final['token_count_analysis'] = df_final['cleaned_tokens_analysis'].str.split().str.len()

print("위치 관련 토큰 제거 후 (df_final의 첫 5개 행):")
display(df_final[['name', 'cleaned_tokens_final', 'token_count_final', 'cleaned_tokens_analysis', 'token_count_analysis']].head())

위치 관련 토큰 제거 후 (df_final의 첫 5개 행):


,name,cleaned_tokens_final,token_count_final,cleaned_tokens_analysis,token_count_analysis
0,메모러블모먼트,둔촌동역 메뉴 블루리본 올림픽 공원 양재대로 둔촌동역 출구 블루리본 맛집 주차 공간...,23,메뉴 블루리본 블루리본 공간 협소 한살림 유료 도보 블루리본 우연히,10
1,메모러블모먼트,분위기 좋다 블루리본 디저트 솔직 아몬드 인상 디저트 메뉴 좋다 분위기 조용하다 깔...,25,분위기 좋다 블루리본 디저트 솔직 아몬드 인상 디저트 메뉴 좋다 분위기 조용하다 깔...,24
2,메모러블모먼트,올림픽 공원 골목 국민 떡볶이 건물 주차장 따로 올림픽 공원 가깝다 저희 올림픽 공...,20,국민 떡볶이 따로 가깝다 저희 산책 프랜차이즈 손님 생각,9
3,메모러블모먼트,블루리본 분위기 좋다 토일 라스트 마감 루시 우연히 발견 블루리본 메뉴 생각 다양하...,22,블루리본 분위기 좋다 토일 라스트 마감 루시 우연히 발견 블루리본 메뉴 생각 다양하...,22
4,메모러블모먼트,찰나 순간 기록 솔직 이미 감성 소문 자자하다 참새 방앗간 무조건 양재대로 강동역 ...,29,찰나 순간 기록 솔직 이미 감성 소문 자자하다 참새 방앗간 특징 블루리본 베이 연속...,24


In [ ]:
df_final.to_csv('df_final_final.csv', index=False, encoding='utf-8-sig')
print("df_final DataFrame이 'df_final_final.csv'로 저장되었습니다.")

df_final DataFrame이 'df_final_final.csv'로 저장되었습니다.
